# Encoding and Scaling

In [8]:
import numpy as np 
import pandas as pd

df = pd.read_csv("./data/04_Outlar_Treatement.csv")

In [9]:
df.shape

(57130, 44)

# Data

In [10]:
import pandas as pd
import numpy as np

# 1. Dataset overview
print("Dataset Shape:", df.shape)
print("\nData Types:")
print(df.dtypes.value_counts())

# 2. Identify numerical and categorical columns
numerical_features = df.select_dtypes(include=np.number).columns.tolist()

categorical_features = df.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

# Exclude target from feature encoding analysis
target = "construction_cost"

numerical_features = [
    col for col in numerical_features if col != target
]

categorical_features = [
    col for col in categorical_features if col != target
]

print("\nNumerical Features:", len(numerical_features))
print(numerical_features)

print("\nCategorical Features:", len(categorical_features))
print(categorical_features)

# 3. Analyze categorical features
categorical_summary = pd.DataFrame({
    "Feature": categorical_features,
    "Data Type": [df[col].dtype for col in categorical_features],
    "Unique Values": [df[col].nunique() for col in categorical_features],
    "Missing Values": [df[col].isnull().sum() for col in categorical_features],
    "Unique Percentage": [
        round(df[col].nunique() / len(df) * 100, 2)
        for col in categorical_features
    ],
    "Sample Values": [
        df[col].dropna().unique()[:10].tolist()
        for col in categorical_features
    ]
})

# 4. Categorize based on cardinality
categorical_summary["Category"] = categorical_summary[
    "Unique Values"
].apply(
    lambda x: "Binary" if x == 2
    else "Low Cardinality" if x <= 10
    else "High Cardinality"
)

# 5. Display categorical summary
display(
    categorical_summary.sort_values(
        by="Unique Values",
        ascending=False
    ).reset_index(drop=True)
)

# 6. Target variable verification
print("\nTarget Variable:")
print("Name:", target)
print("Data Type:", df[target].dtype)
print("Unique Values:", df[target].nunique())

Dataset Shape: (57130, 44)

Data Types:
float64    23
object     19
int64       2
Name: count, dtype: int64

Numerical Features: 24
['latitude', 'longitude', 's_no_of_scheme', 'total_owners_holding', 'scheme_depth', 'scheme_diameter', 'nearest_scheme_distance', 'machinery_cost', 'maintenance_cost', 'construction_subsidy', 'machinery_subsidy', 'horse_power_of_lifting_device', 'culturable_command_area', 'ipc_kharif_season', 'ipc_rabi_season', 'ipc_perennial_crop', 'ipc_other_season', 'ipc_total', 'ipu_kharif_season', 'ipu_rabi_season', 'ipu_perennial_crop', 'ipu_other_season', 'ipu_total', 'Total_Outlier_Flags']

Categorical Features: 19
['district_name', 'block_tehsil_name', 'village_name', 'scheme_type', 'ownership', 'social_status', 'gender', 'commissioning_year', 'scheme_status', 'water_distribution_method', 'lifting_device_type', 'energy_source', 'located_in_command_area', 'only_recharge_ground_water', 'under_utilised_selection', 'drinking_water_purpose', 'captured_in_previous_censu

,Feature,Data Type,Unique Values,Missing Values,Unique Percentage,Sample Values,Category
0,village_name,object,8356,0,14.63,"[Siluk Pt-Ii, Bhadiaguri, Patiarpara Pt-I, Boa...",High Cardinality
1,block_tehsil_name,object,227,0,0.40,"[Kuchdhowa, Kokrajhar, Nayeralga, Lahoal, Beha...",High Cardinality
2,district_name,object,34,0,0.06,"[Goalpara, Kokrajhar, Dhubri, Dibrugarh, Biswa...",High Cardinality
3,finance_source,object,18,0,0.03,"[Government fund, Own savings, Others, Others,...",High Cardinality
4,water_distribution_method,object,8,0,0.01,"[Other, Open Water Channel (lined / Pucca), Op...",Low Cardinality
5,energy_source,object,8,0,0.01,"[Solar, Electric, Diesel, Manual/animal, Not a...",Low Cardinality
6,well_type,object,8,0,0.01,"[Tubewell: Medium Tube well, Tubewell: Shallow...",Low Cardinality
7,commissioning_year,object,6,0,0.01,"[Up to 2018-2019, During 2021-2022, During 201...",Low Cardinality
8,ownership,object,6,0,0.01,"[Govt. Owned, Owned by individual farmer, Othe...",Low Cardinality
9,lifting_device_type,object,6,0,0.01,"[Others, Submersible pump, Turbine/Jet pump, C...",Low Cardinality



Target Variable:
Name: construction_cost
Data Type: float64
Unique Values: 1270


# Categorical Encoding

## Binary Encoding

In [11]:
# Binary Encoding

binary_mappings = {
    "scheme_type": {
        "Dug well": 0,
        "Tube well": 1
    },
    "located_in_command_area": {
        "No": 0,
        "Yes": 1
    },
    "only_recharge_ground_water": {
        "No": 0,
        "Yes": 1
    },
    "drinking_water_purpose": {
        "No": 0,
        "Yes": 1
    }
}

# Validate categories before encoding
for col, mapping in binary_mappings.items():
    actual_values = set(df[col].dropna().unique())
    expected_values = set(mapping.keys())

    if actual_values != expected_values:
        raise ValueError(
            f"Unexpected values in {col}: {actual_values}"
        )

# Apply binary encoding
for col, mapping in binary_mappings.items():
    df[col] = df[col].map(mapping).astype("int8")

# Verify binary encoding
for col in binary_mappings:
    print(f"\n{col}")
    print(df[col].value_counts().sort_index())

print("\nDataset Shape:", df.shape)


scheme_type
scheme_type
0      794
1    56336
Name: count, dtype: int64

located_in_command_area
located_in_command_area
0    56985
1      145
Name: count, dtype: int64

only_recharge_ground_water
only_recharge_ground_water
0    56007
1     1123
Name: count, dtype: int64

drinking_water_purpose
drinking_water_purpose
0    55365
1     1765
Name: count, dtype: int64

Dataset Shape: (57130, 44)


## Ordinal Encoding

In [12]:
# Inspect commissioning year categories

print("Unique commissioning_year values:")
print(df["commissioning_year"].value_counts())

print("\nUnique categories:")
print(df["commissioning_year"].unique().tolist())

Unique commissioning_year values:
commissioning_year
Up to 2018-2019     22140
During 2019-2020    11717
During 2020-2021     9591
During 2021-2022     7116
During 2022-2023     3525
During 2023-2024     3041
Name: count, dtype: int64

Unique categories:
['Up to 2018-2019', 'During 2021-2022', 'During 2019-2020', 'During 2022-2023', 'During 2020-2021', 'During 2023-2024']


In [13]:
# Ordinal Encoding for commissioning_year

commissioning_order = [
    "Up to 2018-2019",
    "During 2019-2020",
    "During 2020-2021",
    "During 2021-2022",
    "During 2022-2023",
    "During 2023-2024"
]

commissioning_mapping = {
    category: index
    for index, category in enumerate(commissioning_order)
}

# Validate categories
actual_values = set(df["commissioning_year"].dropna().unique())
expected_values = set(commissioning_mapping.keys())

if actual_values != expected_values:
    raise ValueError("Unexpected commissioning_year categories!")

# Apply ordinal encoding
df["commissioning_year"] = (
    df["commissioning_year"]
    .map(commissioning_mapping)
    .astype("int8")
)

# Verify encoding
print("Encoded commissioning_year:")
print(df["commissioning_year"].value_counts().sort_index())

print("\nData Type:", df["commissioning_year"].dtype)
print("Dataset Shape:", df.shape)

Encoded commissioning_year:
commissioning_year
0    22140
1    11717
2     9591
3     7116
4     3525
5     3041
Name: count, dtype: int64

Data Type: int8
Dataset Shape: (57130, 44)


### One-Hot Encoding

In [14]:
# One-Hot Encoding

onehot_features = [
    "water_distribution_method",
    "energy_source",
    "well_type",
    "ownership",
    "lifting_device_type",
    "social_status",
    "gender",
    "scheme_status",
    "captured_in_previous_census",
    "under_utilised_selection",
    "finance_source"
]

# Verify that all selected columns exist
missing_columns = [
    col for col in onehot_features if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

# Verify no unexpected missing values
for col in onehot_features:
    if df[col].isnull().any():
        raise ValueError(f"Missing values found in {col}")

# Apply One-Hot Encoding
df = pd.get_dummies(
    df,
    columns=onehot_features,
    drop_first=False,
    dtype="int8"
)

# Verify results
print("Dataset Shape After One-Hot Encoding:", df.shape)

print("\nRemaining Categorical Columns:")
print(df.select_dtypes(include=["object", "category"]).columns.tolist())

print("\nGenerated One-Hot Columns:")
print([
    col for col in df.columns
    if any(col.startswith(feature + "_") for feature in onehot_features)
])

Dataset Shape After One-Hot Encoding: (57130, 105)

Remaining Categorical Columns:
['district_name', 'block_tehsil_name', 'village_name']

Generated One-Hot Columns:
['water_distribution_method_Drip', 'water_distribution_method_Not applicable', 'water_distribution_method_Open Water Channel (lined / Pucca)', 'water_distribution_method_Open Water Channel (unlined / Kutcha)', 'water_distribution_method_Other', 'water_distribution_method_Sprinkler', 'water_distribution_method_Surface pipe', 'water_distribution_method_Under ground pipe', 'energy_source_Diesel', 'energy_source_Electric', 'energy_source_Manual/animal', 'energy_source_Natural', 'energy_source_Not applicable', 'energy_source_Others', 'energy_source_Solar', 'energy_source_Wind Mills', 'well_type_Dugwell: Dug well Kutcha', 'well_type_Dugwell: Dug well Pucca', 'well_type_Dugwell: Dug-cum-bore well', 'well_type_Dugwell: Others', 'well_type_Tubewell: Artesian Well', 'well_type_Tubewell: Deep Tube well', 'well_type_Tubewell: Medium T

## Frequency Encoding

In [15]:
# Frequency Encoding

frequency_features = [
    "district_name",
    "block_tehsil_name",
    "village_name"
]

# Store mappings for future unseen-category handling
frequency_mappings = {}

for col in frequency_features:

    # Calculate normalized category frequencies
    frequency_map = df[col].value_counts(normalize=True)

    # Store mapping
    frequency_mappings[col] = frequency_map.to_dict()

    # Apply frequency encoding
    df[col] = df[col].map(frequency_map).astype("float64")

# Verify frequency encoding
for col in frequency_features:
    print(f"\n{col}")
    print(df[col].describe())

print("\nDataset Shape:", df.shape)

print("\nRemaining Categorical Columns:")
print(df.select_dtypes(include=["object", "category"]).columns.tolist())


district_name
count    57130.000000
mean         0.119940
std          0.100399
min          0.001085
25%          0.022230
50%          0.087520
75%          0.259723
max          0.259723
Name: district_name, dtype: float64

block_tehsil_name
count    57130.000000
mean         0.018144
std          0.016234
min          0.000018
25%          0.004166
50%          0.012690
75%          0.025783
max          0.059916
Name: block_tehsil_name, dtype: float64

village_name
count    57130.000000
mean         0.000414
std          0.000489
min          0.000018
25%          0.000123
50%          0.000263
75%          0.000508
max          0.003868
Name: village_name, dtype: float64

Dataset Shape: (57130, 105)

Remaining Categorical Columns:
[]


## Verify Encoded Data

In [16]:
# Encoding Verification

print("=" * 60)
print("ENCODING VERIFICATION")
print("=" * 60)

# 1. Dataset shape
print("\n1. Dataset Shape:")
print(df.shape)

# 2. Data type distribution
print("\n2. Data Type Distribution:")
print(df.dtypes.value_counts())

# 3. Remaining categorical columns
remaining_categorical = df.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\n3. Remaining Categorical Columns:")
print(remaining_categorical)

# 4. Missing values
print("\n4. Missing Values:")
print("Total Missing Values:", df.isnull().sum().sum())

if df.isnull().sum().sum() > 0:
    display(
        df.isnull().sum()
        .loc[lambda x: x > 0]
        .to_frame("Missing Values")
    )

# 5. Target variable verification
print("\n5. Target Variable:")
print("Target:", "construction_cost")
print("Data Type:", df["construction_cost"].dtype)
print("Missing Values:", df["construction_cost"].isnull().sum())

# 6. Duplicate columns
print("\n6. Duplicate Column Names:")
print(df.columns[df.columns.duplicated()].tolist())

# 7. Numerical columns
print("\n7. Numerical Columns:", len(
    df.select_dtypes(include=np.number).columns
))

# 8. Final status
print("\n8. Encoding Status:")

if len(remaining_categorical) == 0:
    print("All categorical columns have been encoded.")
else:
    print("Some categorical columns remain:", remaining_categorical)

print("\nVerification Completed.")

ENCODING VERIFICATION

1. Dataset Shape:
(57130, 105)

2. Data Type Distribution:
int8       77
float64    26
int64       2
Name: count, dtype: int64

3. Remaining Categorical Columns:
[]

4. Missing Values:
Total Missing Values: 0

5. Target Variable:
Target: construction_cost
Data Type: float64
Missing Values: 0

6. Duplicate Column Names:
[]

7. Numerical Columns: 105

8. Encoding Status:
All categorical columns have been encoded.

Verification Completed.


# Save Dataset

In [18]:
df.shape

(57130, 105)

In [ ]:
df.to_csv("./data/05_Encoding.csv", index=False)